# Querying File

## 1.1 Check Sample Data Source from Databricks

In [0]:
%python
display(dbutils.fs.ls("dbfs:/databricks-datasets"))

In [0]:
%python
display(dbutils.fs.ls("/Volumes/aman/bookstore/data/"))

In [0]:
%sql
select * from json.`/Volumes/aman/bookstore/data/customers-json/`
where customer_id = 'C00001'

we can directly query file using file format and path
select * from file_format.path

## 1.2 Querying File Directly from File Format

### 1.2.1 JSON

In [0]:
%sql
select * from json.`/Volumes/aman/bookstore/data/customers-json/`


### 1.2.2 Text

In [0]:
%sql
select * from text.`dbfs:/databricks-datasets/README.md`

## 2. Create Table using DATA 

### 2.1 CTAS

In CTAS statement, we can't infer schema

In [0]:
create table customer_ingested
as 
select * from json.`/Volumes/aman/bookstore/data/customers-json/`


In [0]:
select * from customer_ingested

In [0]:
describe extended customer_ingested

### 2.2 Create Table Statement

we can create well defined delta table using create table statement with defined schema using data sources

In [0]:
select * from csv.`/Volumes/aman/bookstore/data/books-csv/`

In [0]:
create schema aman.delta

we can create table and ingest data using COPY INTO activity

In [0]:
-- create table
CREATE OR REPLACE TABLE aman.delta.customer_2
(
    book_id STRING,
    title STRING,
    author STRING,
    category STRING,
    price INT
);
-- copy data into the table
COPY INTO aman.delta.customer_2
FROM '/Volumes/aman/bookstore/data/books-csv/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'delimiter' = ';', 'inferSchema' = 'true')

In [0]:
describe extended aman.delta.customer_2

In [0]:
select * from aman.delta.customer_2

### 2.3 Create Table from External Location

The Location need to refer external cloud storage location but the table created using this command is not delta

## VIP:-The code below doesn't work as we don't have external cloud file location

In [0]:
CREATE TABLE aman.delta.customer_2
(
    book_id STRING,
    title STRING,
    author STRING,
    category STRING,
    price INT
)
USING CSV
OPTIONS (
    header = 'true',
    delimiter = ';'
)
LOCATION '/Volumes/aman/bookstore/data/books-csv/';

In order to change the table into delta format, we can create temporary view and create table using CTAS statement from view

In [0]:
create temporary view customer_2_1 
(
    book_id STRING,
    title STRING,
    author STRING,
    category STRING,
    price INT
)
USING CSV
OPTIONS (
    header = 'true',
    delimiter= ';',
    path = '/Volumes/aman/bookstore/data/books-csv/'
)


In [0]:
select * from customer_2_1

In [0]:
create table aman.delta.customer_2_1
as 
select * from customer_2_1

In [0]:
select * from aman.delta.customer_2_1

## Eg. Create Order Table

In [0]:
create or replace table orders as 
select * from parquet.`dbfs:/Volumes/aman/bookstore/data/orders/`

In [0]:
select * from orders

In [0]:
%python 
from pyspark.sql.functions import *

###Note: We want to overwrite and insert new data but it failed due to our original order table don't have the new metadata information in creation time

In [0]:
Insert overwrite orders

select *,
_metadata.file_path as file_path,
current_timestamp as Registered_time,
_metadata.file_size as size,
_metadata.file_name as file_name

from parquet.`dbfs:/Volumes/aman/bookstore/data/orders/`

### Replace our table with new format of enriched metadat information

In [0]:
Create or replace table  orders
select *,
_metadata.file_path as file_path,
current_timestamp as Registered_time,
_metadata.file_size as size,
_metadata.file_name as file_name

from parquet.`dbfs:/Volumes/aman/bookstore/data/orders/`




In [0]:
select * from orders

### We can apply insert overwrite to overwrite existing data

In [0]:
Insert overwrite orders

select *,
_metadata.file_path as file_path,
current_timestamp as Registered_time,
_metadata.file_size as size,
_metadata.file_name as file_name

from parquet.`dbfs:/Volumes/aman/bookstore/data/orders/`

### The number of written rows is the same as Insert overwrite used to overwite our data

## 3. Upsert

In [0]:
%python
display(dbutils.fs.ls("/Volumes/aman/bookstore/data/"))

Upsert is the combination of insert, delete and update

##Eg. Original Table :- Customer

In [0]:
create or replace table customer
as 
select * from json.`/Volumes/aman/bookstore/data/customers-json/`


In [0]:
select * from customer 
where email is null

## Source: Customer Update

In [0]:
create or replace table customer_updated
as 
select * from json.`dbfs:/Volumes/aman/bookstore/data/customers-json-new/`

In [0]:
select * from customer_updated

### What is Our Task?

## 1. we need to update an empty email data in customer table with data from customer update
## 2. We need to insert new data which is found in updated table (customer_updated) but not in original (customer) table 

####Check_Point 1:- Unique Id which is found in updated table but not in original table

In [0]:
select 
      customer_id
from customer_updated
where customer_id not in (
    select 
         customer_id
    from customer
)

#### Note 1: This is records which found in Customer_updated table but not in customer table, so they will get inserted

#### Check Point 2: Customer Table with Empty email data but updated table do have email data

In [0]:
select 
     c.customer_id,
     c.email,
     u.email as updated_email
 from customer c
inner join customer_updated u
on c.customer_id = u.customer_id
where c.email is null

#### Note 2: The above 100 rows in with email field will be populated with updated_email from Customer updated

In [0]:
merge into customer c
using customer_updated u 
on c.customer_id = u.customer_id
when matched and c.email is null and u.email is not null
 then update set c.email = u.email 
 when not matched then insert * ;

### VIP : The Upsert statment above updated 100 record in email field and insert 201 new records

### Let Us Check if the new row get inserted if we get 0 records, we are correct

In [0]:
select 
      customer_id
from customer_updated
where customer_id not in (
    select 
         customer_id
    from customer
)

In [0]:
describe history customer;

In the first check point, customer_id C01702 didn't found in customer table, our below query results shows we already insert this data into our customer table

In [0]:
select * from customer
where customer_id = 'C01702'

The customer_id ,C00024, don't have email in customer table but now get updated with the new email

In [0]:
select * from customer
where customer_id = 'C00024'